In [7]:
import numpy as np
import pandas as pd

In [8]:
def extract_stability_features_per_rep(rep_df):
    """
    Takes a dataframe containing exactly ONE repetition and calculates
    all 10 required Bar Path, Drift & Stability summary features.
    """
    # Find the frame index representing the absolute bottom of the squat
    bottom_idx = rep_df['hip_y_smooth'].idxmax()
    bottom_frame = rep_df.loc[bottom_idx]
    
    # Starting positions of the rep (first row of this rep slice)
    start_frame = rep_df.iloc[0]

    # --- 1. Standard Deviations of X coordinates (Your original metrics) ---
    # Captures total horizontal swaying/jitter across the whole repetition
    shoulder_x_sd = rep_df['shoulder_x_smooth'].std()
    hip_x_sd = rep_df['hip_x_smooth'].std()
    knee_x_sd = rep_df['knee_x_smooth'].std()

    # --- 2. Horizontal Correlations (Your original metrics) ---
    # Evaluates how well joints are synchronized. 
    # fillna(0) prevents issues if a user keeps their joints perfectly locked (zero variance)
    shoulder_hip_x_corr = rep_df['shoulder_x_smooth'].corr(rep_df['hip_x_smooth'])
    hip_knee_x_corr = rep_df['hip_x_smooth'].corr(rep_df['knee_x_smooth'])
    
    shoulder_hip_x_corr = 0.0 if np.isnan(shoulder_hip_x_corr) else shoulder_hip_x_corr
    hip_knee_x_corr = 0.0 if np.isnan(hip_knee_x_corr) else hip_knee_x_corr

    # --- 3. Upgraded Absolute X Drifts ---
    # Measures the maximum horizontal deviation from where they started the rep
    shoulder_max_x_drift = rep_df['shoulder_x_smooth'].apply(lambda x: abs(x - start_frame['shoulder_x_smooth'])).max()
    hip_max_x_drift = rep_df['hip_x_smooth'].apply(lambda x: abs(x - start_frame['hip_x_smooth'])).max()

    # --- 4. Ratios scaled by depth ---
    # We define vertical depth baseline as the hip vertical travel from start to bottom
    vertical_travel_depth = bottom_frame['hip_y_smooth'] - start_frame['hip_y_smooth']
    if vertical_travel_depth == 0:
        vertical_travel_depth = 0.001 # Prevent division by zero

    # Torso lean metric proxy at bottom frame scaled by vertical travel depth
    torso_lean_to_depth_ratio = (bottom_frame['shoulder_x_smooth'] - bottom_frame['hip_x_smooth']) / vertical_travel_depth
    
    # Knee horizontal travel from start to bottom scaled by depth
    knee_x_travel = abs(bottom_frame['knee_x_smooth'] - start_frame['knee_x_smooth'])
    knee_travel_to_depth_ratio = knee_x_travel / vertical_travel_depth

    # --- 5. Ascent Path Symmetry ---
    # Compares the path taken down vs. the path taken up. 
    # Perfect control yields highly correlated paths. Striking structural shifts break this.
    descent_df = rep_df.loc[:bottom_idx]
    ascent_df = rep_df.loc[bottom_idx:]
    
    # Interpolate down/up phases to same length to compute correlation across the phases
    descent_x = np.interp(np.linspace(0, 1, 50), np.linspace(0, 1, len(descent_df)), descent_df['hip_x_smooth'].values)
    ascent_x = np.interp(np.linspace(0, 1, 50), np.linspace(0, 1, len(ascent_df)), ascent_df['hip_x_smooth'].values)
    
    # Flip the ascent array so chronological mapping matches spatial tracking points
    ascent_x_reversed = np.flip(ascent_x)
    
    ascent_path_symmetry = np.corrcoef(descent_x, ascent_x_reversed)[0, 1]
    ascent_path_symmetry = 0.0 if np.isnan(ascent_path_symmetry) else ascent_path_symmetry

    return {
        'shoulder_x_sd': shoulder_x_sd,
        'hip_x_sd': hip_x_sd,
        'knee_x_sd': knee_x_sd,
        'shoulder_hip_x_corr': shoulder_hip_x_corr,
        'hip_knee_x_corr': hip_knee_x_corr,
        'shoulder_max_x_drift': shoulder_max_x_drift,
        'hip_max_x_drift': hip_max_x_drift,
        'torso_lean_to_depth_ratio': torso_lean_to_depth_ratio,
        'knee_travel_to_depth_ratio': knee_travel_to_depth_ratio,
        'ascent_path_symmetry': ascent_path_symmetry
    }